# 01 - Build Dataset
Construct the table-leg removal dataset from PartNet raw mesh data.

**Run cells one by one.** Cell 3 tests on a single sample first to catch problems before the full build.

In [1]:
import sys, os, gc, json
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Increase recursion limit for deep PartNet JSON trees
sys.setrecursionlimit(5000)

from src.config import load_config, ensure_dirs
print('Modules loaded OK')

Modules loaded OK


In [2]:
# Load configuration
cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'table_two_leg.yaml'))
ensure_dirs(cfg)

PARTNET_ROOT = cfg['paths']['partnet_root']
OUTPUT_DIR   = cfg['paths']['raw_data_dir']

print(f'PartNet root : {PARTNET_ROOT}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Category     : {cfg["dataset"]["category"]}')
print(f'Remove part  : {cfg["dataset"]["semantic_label"]}')
print(f'Total samples: {cfg["dataset"]["total_samples"]}')

# Sanity check PartNet path
if not os.path.isdir(PARTNET_ROOT):
    raise FileNotFoundError(
        f'PartNet directory not found: {PARTNET_ROOT}\n'
        f'Edit configs/default.yaml -> paths.partnet_root'
    )

subdirs = [d for d in os.listdir(PARTNET_ROOT)
           if os.path.isdir(os.path.join(PARTNET_ROOT, d))]
print(f'\nPartNet has {len(subdirs)} subdirectories')
print(f'First 5: {subdirs[:5]}')

PartNet root : E:/datasets/partnet
Output dir   : D:/MyJupyter/Works/3DPART_v2/data/raw/table_two_leg_removal
Category     : Table
Remove part  : leg
Total samples: 20

PartNet has 32537 subdirectories
First 5: ['1000', '10000', '10003', '10005', '10007']


In [3]:
# ============================================================
# STEP 1: Quick scan to find Table samples with "leg" parts
#         (no mesh loading, only reads JSON - safe & fast)
# ============================================================
from src.data.dataset_builder import DatasetBuilder

builder = DatasetBuilder(
    partnet_root=PARTNET_ROOT,
    output_dir=OUTPUT_DIR,
    category=cfg['dataset']['category'],
    semantic_label=cfg['dataset']['semantic_label'],
    total_samples=cfg['dataset']['total_samples'],
    random_seed=cfg['dataset']['random_seed'],
    num_parts_to_remove=cfg['dataset'].get('num_parts_to_remove', 1),
    part_selection_mode=cfg['dataset'].get('part_selection_mode', 'random_distinct'),
)

print('Scanning for candidate samples (JSON only, no mesh loading)...')
candidates = builder._find_partnet_samples()
print(f'Found {len(candidates)} candidate {cfg["dataset"]["category"]} samples with {cfg["dataset"]["semantic_label"]} parts')

if len(candidates) > 0:
    print(f'First 5 candidate IDs: {[os.path.basename(c) for c in candidates[:5]]}')
else:
    print('ERROR: No candidates found. Check PartNet directory structure.')

Scanning for candidate samples (JSON only, no mesh loading)...


2026-04-16 13:46:25 [INFO] DatasetBuilder: Scan complete: 7995 valid, 22631 skipped                                    


Found 7995 candidate Table samples with leg parts
First 5 candidate IDs: ['18142', '18143', '18156', '18157', '18158']


In [4]:
# ============================================================
# STEP 2: Test on ONE sample before full build
#         If this cell crashes, the issue is in mesh loading
# ============================================================
from src.io.mesh_io import load_mesh_lightweight, merge_arrays, save_mesh
from pathlib import Path

test_dir = candidates[0]
test_id = os.path.basename(test_dir)
print(f'Testing on sample: {test_id}')

# Parse annotation
anno_file = os.path.join(test_dir, 'result_after_merging.json')
if not os.path.exists(anno_file):
    anno_file = os.path.join(test_dir, 'result.json')

with open(anno_file, 'r', encoding='utf-8') as f:
    anno = json.load(f)

part_groups = builder._find_part_objs(anno, cfg['dataset']['semantic_label'])
all_obj_ids = builder._collect_all_obj_ids(anno)
print(f'  Total OBJ IDs: {len(all_obj_ids)}')
print(f'  {cfg["dataset"]["semantic_label"].capitalize()} part groups: {len(part_groups)}')
print(f'  First group has {len(part_groups[0])} OBJs: {part_groups[0][:5]}')

# Test loading a single OBJ
objs_dir = os.path.join(test_dir, 'objs')
test_obj = os.path.join(objs_dir, f'{all_obj_ids[0]}.obj')
print(f'\n  Test loading: {test_obj}')
md = load_mesh_lightweight(test_obj)
if md is not None:
    print(f'  OK: {md["vertices"].shape[0]} vertices, {md["faces"].shape[0]} faces')
    del md
else:
    print('  FAILED to load. Check OBJ file format.')

gc.collect()
print('\n  Single OBJ test passed!')

Testing on sample: 18142
  Total OBJ IDs: 45
  Leg part groups: 1
  First group has 16 OBJs: ['original-3', 'original-2', 'original-5', 'original-4', 'new-0']

  Test loading: E:\datasets\partnet\18142\objs\original-1.obj
  OK: 8 vertices, 24 faces

  Single OBJ test passed!


In [5]:
# ============================================================
# STEP 3: Process the test sample end-to-end
# ============================================================
import random
random.seed(42)

remove_group = random.choice(part_groups)
remove_set = set(remove_group)

removed_paths = []
remaining_paths = []
for obj_id in all_obj_ids:
    obj_file = os.path.join(objs_dir, f'{obj_id}.obj')
    if not os.path.exists(obj_file):
        continue
    if obj_id in remove_set:
        removed_paths.append(obj_file)
    else:
        remaining_paths.append(obj_file)

print(f'  Removed part OBJs: {len(removed_paths)}')
print(f'  Remaining OBJs:    {len(remaining_paths)}')

# Load removed part
removed_dicts = [load_mesh_lightweight(p) for p in removed_paths]
removed_dicts = [d for d in removed_dicts if d is not None]
removed_mesh = merge_arrays(removed_dicts)
print(f'  Removed mesh: {len(removed_mesh.vertices)} verts, {len(removed_mesh.faces)} faces')
del removed_dicts

# Load remaining parts
remaining_dicts = [load_mesh_lightweight(p) for p in remaining_paths]
remaining_dicts = [d for d in remaining_dicts if d is not None]
damaged_mesh = merge_arrays(remaining_dicts)
print(f'  Damaged mesh: {len(damaged_mesh.vertices)} verts, {len(damaged_mesh.faces)} faces')
del remaining_dicts

# Save test output
test_out = os.path.join(OUTPUT_DIR, '_test_sample')
os.makedirs(test_out, exist_ok=True)
save_mesh(damaged_mesh, os.path.join(test_out, 'damaged.obj'))
save_mesh(removed_mesh, os.path.join(test_out, 'removed_part.obj'))

del damaged_mesh, removed_mesh
gc.collect()

print(f'\n  Single sample test PASSED! Files saved to {test_out}')
print('  Safe to proceed with full build.')

  Removed part OBJs: 40
  Remaining OBJs:    5
  Removed mesh: 81170 verts, 208240 faces
  Damaged mesh: 40 verts, 120 faces

  Single sample test PASSED! Files saved to D:/MyJupyter/Works/3DPART_v2/data/raw/table_two_leg_removal\_test_sample
  Safe to proceed with full build.


In [6]:
# ============================================================
# STEP 4: Full dataset build (uses memory-safe builder)
# ============================================================

# Clean up test
import shutil
test_out = os.path.join(OUTPUT_DIR, '_test_sample')
if os.path.exists(test_out):
    shutil.rmtree(test_out)

gc.collect()

# Rebuild the builder with fresh state
builder = DatasetBuilder(
    partnet_root=PARTNET_ROOT,
    output_dir=OUTPUT_DIR,
    category=cfg['dataset']['category'],
    semantic_label=cfg['dataset']['semantic_label'],
    total_samples=cfg['dataset']['total_samples'],
    random_seed=cfg['dataset']['random_seed'],
)

output_path = builder.build()
print(f'\nDataset saved to: {output_path}')

2026-04-16 13:46:26 [INFO] DatasetBuilder: Scanning PartNet for Table with 'leg' parts...
2026-04-16 13:46:33 [INFO] DatasetBuilder: Scan complete: 7995 valid, 22631 skipped                                    
2026-04-16 13:46:33 [INFO] DatasetBuilder: Selected 20 samples for dataset
Building dataset: 100%|████████████████████████████████████████████████████████████████| 20/20 [00:24<00:00,  1.23s/it]
2026-04-16 13:46:57 [INFO] DatasetBuilder: Dataset built: 20/20 samples saved to D:\MyJupyter\Works\3DPART_v2\data\raw\table_two_leg_removal



Dataset saved to: D:\MyJupyter\Works\3DPART_v2\data\raw\table_two_leg_removal


In [7]:
# Verify dataset
from src.data.dataset_index import DatasetIndex

index = DatasetIndex(OUTPUT_DIR)
print(f'Total samples in dataset: {len(index)}')
print(f'\nFirst 10 sample IDs:')
for sid in index.sample_ids[:10]:
    meta_path = os.path.join(OUTPUT_DIR, sid, 'meta.json')
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        print(f'  {sid}: {meta["mesh_before"]["n_vertices"]} -> '
              f'{meta["mesh_after"]["n_vertices"]} verts '
              f'(removed {len(meta["removed_obj_files"])} OBJs)')

Total samples in dataset: 20

First 10 sample IDs:
  24270: 100390 -> 91225 verts (removed 5 OBJs)
  32881: 1980 -> 1730 verts (removed 5 OBJs)
  29101: 40050 -> 80 verts (removed 25 OBJs)
  26430: 53980 -> 39735 verts (removed 60 OBJs)
  30908: 73805 -> 55080 verts (removed 40 OBJs)
  25724: 54420 -> 51700 verts (removed 80 OBJs)
  23706: 110047 -> 73939 verts (removed 44 OBJs)
  19117: 79875 -> 68225 verts (removed 5 OBJs)
  25445: 1640 -> 40 verts (removed 200 OBJs)
  33988: 42935 -> 38345 verts (removed 20 OBJs)


In [8]:
# Create train/val/test splits
splits = index.save_splits(cfg['paths']['splits_dir'])
for name, ids in splits.items():
    print(f'  {name}: {len(ids)} samples')
print('\nDone! Proceed to notebook 02.')

  train: 14 samples
  val: 3 samples
  test: 3 samples

Done! Proceed to notebook 02.
